# Manipulated AADHAAR ID's Dataset Generation

### Dataset Structure
- **Real images**: Clean cards rendered from genuine-looking data
- **Fake images**: Same cards with one deliberate manipulation applied per variant
- **Face types**: Real, Synthetic, and Morphed faces used across the dataset
- **Output**: ~2800+ images with accompanying json description

In [ ]:
# Mounting google drive that contains the asset folder and output files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Root directory on Google Drive containing all project assets:
ASSET_DIR = "/content/drive/MyDrive/aadhar_project/assets"

In [ ]:
!ls /content/drive/MyDrive/aadhar_project/assets

### Field Coordinates on the Card Template

Each Aadhaar card field (name, address, DOB, ID number, etc.) is printed at a specific pixel location on the template image. The `RAW_COORDS` list maps to the `FIELDS` list in the exact same order.

These coordinates were manually measured from the card template to ensure accurate text placement.

In [ ]:
RAW_COORDS = [[279, 157], [279, 126],
 [278, 223],[413, 191],
 [899, 134], [901, 236],
 [1124, 409], [268, 447],
 [1125, 455], [43, 153],
  [215, 59], [1038, 61],
 [1564, 75], [993, 504],
[1207, 504],[1483, 504],
 [877, 184], [78,118]]


In [ ]:
FIELDS = [
    "name_eng","name_hin",
    "gender", "dob",
    "address_hin", "address_eng",
    "id_number", "id_number_2", "vid",
    "issue_date", "government", "authority", "aadhaar",
    "telephone","email_label",
    "website_label", "details_date", "face"
]

if len(RAW_COORDS) != len(FIELDS):
    raise ValueError("Number of coordinates does not match number of fields")

COORDS = {
    FIELDS[i]: tuple(RAW_COORDS[i])
    for i in range(len(FIELDS))
}


### Synthetic Data Generation
We generate entirely **synthetic Indian identity records** using the `Faker` library with the `hi_IN` (Hindi/India) locale.

Each generated record includes:
- Name in Hindi (Devanagari script) → transliterated to English
- Gender (randomly assigned)
- Date of Birth (age 22–34)
- Indian address in Hindi → transliterated to English
- 12-digit Aadhaar-format ID number
- Issue and details dates

In [ ]:
pip install faker

In [ ]:
!pip install indic-transliteration


In [ ]:
from faker import Faker
import pandas as pd
import random
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
import re

def normalize_english(text: str) -> str:
    text = text.lower()
    # --- known transliteration artifacts ---
    text = text.replace("z", "sh")
    # Basic phonetic smoothing
    text = text.replace("aa", "a").replace("ii", "i").replace("uu", "u")
    text = text.replace("sh", "sh").replace("ch", "ch").replace("th", "th")
    # Remove strange characters
    text = re.sub(r"[^a-z0-9 ,.-]", "", text)
    # Capitalize properly
    return " ".join(word.capitalize() for word in text.split())


def to_english(hindi_text: str) -> str:
    raw = transliterate(hindi_text, sanscript.DEVANAGARI, sanscript.HK)
    return normalize_english(raw)

fake_hi = Faker("hi_IN")

def generate_data(n=50):
    rows = []

    for _ in range(n):
        issue_date = fake_hi.date_between(start_date='-28y', end_date='-15y')
        details_date = fake_hi.date_between(start_date='-8y', end_date='-2y')

        gender_hin = random.choice(["पुरुष", "महिला"])
        gender_eng = "MALE" if gender_hin == "पुरुष" else "FEMALE"
        gender_combined = f"{gender_hin} / {gender_eng}"

        # --- Hindi identity (source of truth) ---
        if gender_hin == "पुरुष":
            first = fake_hi.first_name_male()
        else:
            first = fake_hi.first_name_female()

        last = fake_hi.last_name()

        name_hin = f"{first} {last}"
        name_eng = to_english(name_hin)

        address_hin = fake_hi.address().replace("\n", ", ")
        address_eng = to_english(address_hin)

        rows.append({
            "name_eng": name_eng,
            "name_hin": name_hin,
            "gender": gender_combined,
            "dob": fake_hi.date_of_birth(minimum_age=22, maximum_age=34).strftime("%d/%m/%Y"),
            "issue_date": issue_date.strftime("%d/%m/%Y"),
            "details_date": details_date.strftime("%d/%m/%Y"),
            "address_eng": address_eng,
            "address_hin": address_hin,
            "id_number": str(random.randint(2, 9)) + str(random.randint(10**10, 10**11 - 1))
        })

    return pd.DataFrame(rows)


In [ ]:
# Generating 10k records of credentials
df = generate_data(10000)
print(df.head())


### Defining different manipulation cases

In [ ]:
# Static fields that appear on every authentic Aadhaar card.
# These are the 'official' values; manipulations will deviate from these.

TEMPLATE_BASE = {
    "government": "Government of India",
    "aadhaar": "AADHAAR",
    "telephone": "1947",
    "email_label": "help@uidai.gov.in",
    "website_label": "www.uidai.gov.in",
    "vid": "1234 5678 9012 3456",
    "authority": "Unique Identification Authority of India"
}

import random
from datetime import datetime, timedelta

def random_invalid_date(start_year=1990, end_year=2025):
    year = random.randint(start_year, end_year)

    invalid_cases = [
        # Feb invalid (29–31 Feb)
        lambda: f"{random.randint(29,31):02d}/02/{year}",
        # 31st in 30-day months
        lambda: f"31/{random.choice([4,6,9,11]):02d}/{year}",
        # 30 Feb
        lambda: f"30/02/{year}",
        # Non-leap Feb 29
        lambda: f"29/02/{random.choice([y for y in range(start_year, end_year+1) if y % 4 != 0])}"
    ]

    return random.choice(invalid_cases)()


def random_old_issue_date():
    start = datetime(1995, 1, 1)
    end = datetime(2004, 12, 31)
    delta = end - start
    random_day = random.randint(0, delta.days)
    return (start + timedelta(days=random_day)).strftime("%d/%m/%Y")


MANIPULATIONS = [
    {"label": "clean", "apply": lambda r: r},

    {"label": "gov_spelling",
     "apply": lambda r: {**r, "government": "Govenrment of India"}},

    {"label": "doc_spelling",
     "apply": lambda r: {**r, "aadhaar": "ADHAR"}},

    {"label": "phone_change",
     "apply": lambda r: {**r, "telephone": "1200"}},

    {"label": "email_change",
     "apply": lambda r: {**r, "email_label": "lomer@gmail.com"}},

    {"label": "website_change",
     "apply": lambda r: {**r, "website_label": "www.aadhar.com"}},

    {"label": "invalid_dob",
     "apply": lambda r: {**r, "dob": random_invalid_date()}},

    {"label": "gender_mismatch",
     "apply": lambda r: {
         **r,
         "gender": "महिला / FEMALE" if r["gender"].startswith("पुरुष")
                   else "पुरुष / MALE"
     }},

    {"label": "invalid_id",
     "apply": lambda r: {
         **r,
         "id_number": "1" + str(random.randint(10**11, 10**12 - 1))
     }},

    {"label": "invalid_issue_date",
     "apply": lambda r: {**r, "issue_date": random_invalid_date()}},

    {"label": "invalid_details_date",
     "apply": lambda r: {**r, "details_date": random_invalid_date()}},

    {"label": "authority",
     "apply": lambda r: {**r, "authority": "Identification Unique Authority of India"}}
]



### ID Card Rendering Functions

This section defines all the helper functions that draw text and images onto the Aadhaar card template. The card is loaded as an OpenCV image (BGR format), then converted to a Pillow image (RGB) for text rendering, and finally converted back.

In [ ]:
import cv2
def clear_field(img, x, y, w=260, h=28):
    """Erase a rectangular region of the card by filling it with white.
    Used to wipe existing content before writing new text.
    Args: img (OpenCV BGR image), x/y (top-left pixel), w/h (region size)
    """
    cv2.rectangle(img, (x-2, y-h), (x+w, y+4), (255,255,255), -1)


def apply_one_random_manipulation(record):
    """Pick a random manipulation from MANIPULATIONS and apply it to a record.
    Returns the modified record and the manipulation label.
    """
    m = random.choice(MANIPULATIONS)
    new_record = m["apply"](record)
    return new_record, m["label"]

# Rendering functions used for vertical text (issue_date)
def draw_rotated_text(base_img, position, text, font, angle):
    # Measure original text
    dummy = Image.new("RGBA", (1, 1))
    d = ImageDraw.Draw(dummy)
    w, h = d.textbbox((0, 0), text, font=font)[2:]

    # Create padded canvas
    pad = 10
    temp = Image.new("RGBA", (w + pad*2, h + pad*2), (255, 255, 255, 0))
    temp_draw = ImageDraw.Draw(temp)
    temp_draw.text((pad, pad), text, font=font, fill=(0, 0, 0, 255))

    # Rotate
    rotated = temp.rotate(angle, expand=True)

    # Center the rotated text on the target point
    rx, ry = rotated.size
    cx, cy = position
    paste_x = int(cx - rx / 2)
    paste_y = int(cy - ry / 2)

    base_img.paste(rotated, (paste_x, paste_y), rotated)

def draw_semi_bold(draw, position, text, font, fill=(0,0,0)):
    x, y = position
    draw.text((x, y), text, font=font, fill=fill)
    draw.text((x+1, y), text, font=font, fill=fill)

def format_id(num):
    """Format a 12-digit Aadhaar number as 'XXXX XXXX XXXX' (groups of 4).
    Removes any existing spaces first to ensure consistent formatting.
    """
    s = str(num).replace(" ", "")
    return " ".join([s[i:i+4] for i in range(0, len(s), 4)])

def draw_condensed_text(base_img, position, text, font, fill=(0,0,0), squeeze=0.85):
    # Draw text that is vertically compressed to fit tight spaces on the card.

    x, y = position

    dummy = Image.new("RGBA", (1,1))
    d = ImageDraw.Draw(dummy)
    w, h = d.textbbox((0,0), text, font=font)[2:]

    temp = Image.new("RGBA", (w, h), (255,255,255,0))
    td = ImageDraw.Draw(temp)
    td.text((0,0), text, font=font, fill=fill)

    new_h = int(h * squeeze)
    temp = temp.resize((w, new_h), Image.Resampling.LANCZOS)

    base_img.paste(temp, (x, y), temp)

from PIL import Image

FACE_SIZE = (170, 206)   # Target pixel dimensions for face photos on the card

def paste_face(img_pil, face_path):
    """Paste a face photo onto the card at the designated face coordinate.
    The face image is resized to FACE_SIZE before pasting.
    """
    x, y = COORDS["face"]
    face = Image.open(face_path).convert("RGB")
    face = face.resize(FACE_SIZE)
    img_pil.paste(face, (x, y))


import numpy as np
from PIL import Image, ImageDraw, ImageFont

#Font definitions
FONTS = {
    "government": ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 24),
    "authority": ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 24),
    "aadhaar":    ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Bold.ttf", 16),
    "name":       ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 20),
    "dob":        ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 20),
    "address":    ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 20),
    "gender_eng": ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 20),
    "id_number":  ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 36),
    "vid":        ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Bold.ttf", 26),
    "footer":     ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Regular.ttf", 28),
    "issue_date": ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Bold.ttf", 18),
    "details_date": ImageFont.truetype(f"{ASSET_DIR}/MyriadPro-Bold.ttf", 17)
}

HIN_FONTS = {
    "name":    ImageFont.truetype(f"{ASSET_DIR}/kokila.ttf", 26),
    "address": ImageFont.truetype(f"{ASSET_DIR}/kokila.ttf", 26),
    "gender":  ImageFont.truetype(f"{ASSET_DIR}/kokila.ttf", 28),
    "header":  ImageFont.truetype(f"{ASSET_DIR}/kokilab.ttf", 23)
}


In [ ]:
# Render all fields from `record` onto the Aadhaar card template image.

def render_fields(img_cv, record, face_path):
    img_pil = Image.fromarray(cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)

    # ---- Paste face first ----
    paste_face(img_pil, face_path)

    # ---- Government ----
    x, y = COORDS["government"]
    draw.text((x, y), record["government"], font=FONTS["government"], fill=(0,0,0))

    # ---- Authority ----
    x, y = COORDS["authority"]
    draw.text((x, y), record["authority"], font=FONTS["authority"], fill=(0,0,0))

    # ---- Aadhaar (red) ----
    AADHAAR_RED = (196, 30, 58)
    x, y = COORDS["aadhaar"]
    draw_condensed_text(img_pil, (x, y), record["aadhaar"], FONTS["aadhaar"], fill=AADHAAR_RED, squeeze=0.75)

    # ---- Name ----
    x, y = COORDS["name_eng"]
    draw.text((x, y), record["name_eng"], font=FONTS["name"], fill=(0,0,0))

    x, y = COORDS["name_hin"]
    draw.text((x, y+1), record["name_hin"], font=HIN_FONTS["name"], fill=(0,0,0))

    # ---- DOB ----
    x, y = COORDS["dob"]
    draw.text((x, y), record["dob"], font=FONTS["dob"], fill=(0,0,0))

    # ---- Gender ----
    x, y = COORDS["gender"]
    hin, eng = record["gender"].split(" / ")
    # Hindi (shifted up slightly)
    draw.text((x, y-2), hin, font=HIN_FONTS["gender"], fill=(0,0,0))
    # Place English after Hindi
    w = draw.textlength(hin, font=HIN_FONTS["gender"])
    draw.text((x + w + 6, y), f"/ {eng}", font=FONTS["gender_eng"], fill=(0,0,0))

    # ---- Address ----
    x, y = COORDS["address_eng"]
    draw.text((x, y), record["address_eng"], font=FONTS["address"], fill=(0,0,0))

    x, y = COORDS["address_hin"]
    draw.text((x, y), record["address_hin"], font=HIN_FONTS["address"], fill=(0,0,0))

    # ---- Issue Date (vertical) ----
    x, y = COORDS["issue_date"]
    draw_rotated_text(img_pil, (x, y), record["issue_date"], FONTS["issue_date"], angle=90)

    # ---- Details Date (vertical) ----
    x, y = COORDS["details_date"]
    draw_rotated_text(img_pil, (x, y), record["details_date"], FONTS["details_date"], angle=90)

    # ---- Aadhaar Number (two places) ----
    for key in ["id_number", "id_number_2"]:
        x, y = COORDS[key]
        formatted_id = format_id(record["id_number"])
        draw_semi_bold(draw, (x, y), formatted_id, FONTS["id_number"], fill=(0,0,0))

    # ---- VID ----
    x, y = COORDS["vid"]
    draw.text((x, y), f"VID : {record['vid']}", font=FONTS["vid"], fill=(0,0,0))

    # ---- Footer ----
    for field in ["telephone", "email_label", "website_label"]:
        x, y = COORDS[field]
        draw_condensed_text(img_pil, (x, y), record[field], FONTS["footer"], fill=(0,0,0), squeeze=0.7)


    img_cv[:] = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

def gender_from_face(face_name):
    num = int(face_name[1:-4])  # D1.jpg → 1
    return "पुरुष" if num <= 5 else "महिला"

In [ ]:
# Repeatedly generate single-person records until one matches the given gender.
def get_matching_record(face_gender):
    while True:
        person = generate_data(1).iloc[0].to_dict()
        if person["gender"].startswith(face_gender):
            return person


### Generating manipulated dataset by rendering base template
We first render a **baseline clean card** for every real face image (real, synthetic and morphed faces) and then apply all types of manipulation one by one.

In [ ]:
import os
import cv2
import json
import pandas as pd

ASSET_DIR = "/content/drive/MyDrive/aadhar_project/assets"
BASE_OUTPUT = "/content/drive/MyDrive/aadhar_project/generated"

os.makedirs(BASE_OUTPUT, exist_ok=True)

DATASETS = {
    "synthetic_faces": "synthetic",
    "morphed_faces": "morphed",
    "real_faces": "real"
}
GENDERS = ["male", "female"]

template_img = cv2.imread(f"{ASSET_DIR}/aadhaar card template.jpg")

metadata = []
counter = 1

for dataset_key, dataset_name in DATASETS.items():

    dataset_output_dir = os.path.join(BASE_OUTPUT, dataset_name)
    json_output_dir = os.path.join(dataset_output_dir, "json")

    os.makedirs(dataset_output_dir, exist_ok=True)
    os.makedirs(json_output_dir, exist_ok=True)

    for gender_folder in GENDERS:

        face_dir = os.path.join(ASSET_DIR, dataset_key, gender_folder)

        face_files = sorted([
            f for f in os.listdir(face_dir)
            if f.lower().endswith(".jpg")
        ])

        gender_hin = "पुरुष" if gender_folder == "male" else "महिला"
        for face_name in face_files:

            face_path = os.path.join(face_dir, face_name)

            person = get_matching_record(gender_hin)
            record_base = {**TEMPLATE_BASE, **person}

            for manipulation in MANIPULATIONS:

                record = record_base.copy()
                record = manipulation["apply"](record)
                img = template_img.copy()
                render_fields(img, record, face_path)

                filename = f"{gender_folder}_{face_name[:-4]}_{manipulation['label']}.jpg"
                filepath = os.path.join(dataset_output_dir, filename)

                cv2.imwrite(filepath, img)

                json_data = {
                    "file": filename,
                    "dataset": dataset_name,
                    "face": face_name,
                    "gender": gender_folder,
                    "manipulation": manipulation["label"],
                    "data": record
                }

                json_path = os.path.join(
                    json_output_dir,
                    f"{gender_folder}_{face_name[:-4]}_{manipulation['label']}.json"
                )

                with open(json_path, "w", encoding="utf-8") as f:
                    json.dump(json_data, f, ensure_ascii=False, indent=4)

                metadata.append({
                    "file": filename,
                    "dataset": dataset_name,
                    "gender": gender_folder,
                    "manipulation": manipulation["label"]
                })

                counter += 1

labels_df = pd.DataFrame(metadata)
labels_df.to_csv(os.path.join(BASE_OUTPUT, "labels.csv"), index=False)



Dataset for real faces

In [ ]:
import os
import cv2
import json
import pandas as pd
import copy

ASSET_DIR = "/content/drive/MyDrive/aadhar_project/assets"
REAL_CLEAN_JSON_DIR = "/content/drive/MyDrive/aadhar_project/real_clean/json"

OUTPUT_DIR = "/content/drive/MyDrive/aadhar_project/final_dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

template_img = cv2.imread(f"{ASSET_DIR}/aadhaar card template.jpg")

MASTER_JSON = []
metadata = []

json_files = sorted([
    f for f in os.listdir(REAL_CLEAN_JSON_DIR)
    if f.endswith(".json")
])

for json_file in json_files:

    json_path = os.path.join(REAL_CLEAN_JSON_DIR, json_file)

    with open(json_path, "r", encoding="utf-8") as f:
        json_data = json.load(f)


    record_base = json_data["data"]

    face_name = json_data["face"]
    gender = json_data["gender"]

    face_path = os.path.join(
        ASSET_DIR,
        "real_faces",
        gender,
        face_name
    )

    base_name = json_file.replace(".json", "")


    img = template_img.copy()
    render_fields(img, record_base, face_path)

    base_name = base_name.replace("_clean", "")
    clean_filename = f"{base_name}_authority.jpg"
    cv2.imwrite(os.path.join(OUTPUT_DIR, clean_filename), img)

    MASTER_JSON.append({
        "image_name": clean_filename,
        "label": "Real",
        "face_type": "Real",
        "manipulation": "None",
        "all_manipulations": [],
        "data": record_base
    })

    metadata.append({
        "file": clean_filename,
        "label": "Real"
    })


    for manipulation in MANIPULATIONS:
        if manipulation["label"] == "clean":
            continue

        record = copy.deepcopy(record_base)
        record = manipulation["apply"](record)

        img = template_img.copy()
        render_fields(img, record, face_path)

        filename = f"{base_name}_{manipulation['label']}.jpg"
        filepath = os.path.join(OUTPUT_DIR, filename)

        cv2.imwrite(filepath, img)

        MASTER_JSON.append({
            "image_name": filename,
            "label": "Fake",
            "face_type": "Real",
            "manipulation": manipulation["label"],
            "all_manipulations": [m["label"] for m in MANIPULATIONS],
            "data": record
        })

        metadata.append({
            "file": filename,
            "label": "Fake"
        })

# ======================================
#  SAVE MASTER JSON
with open(os.path.join(OUTPUT_DIR, "master_labels.json"), "w", encoding="utf-8") as f:
    json.dump(MASTER_JSON, f, indent=4, ensure_ascii=False)

# ======================================
#  SAVE CSV

pd.DataFrame(metadata).to_csv(
    os.path.join(OUTPUT_DIR, "labels.csv"),
    index=False
)

print(" Dataset created ONLY from JSON (no image reuse)")
print(f"Total images: {len(MASTER_JSON)}")

In [ ]:
from google.colab import files
import shutil

folder_path = "/content/drive/MyDrive/aadhar_project/final_dataset"
zip_path = "/content/final_dataset.zip"

# Create zip temporarily in /content
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', folder_path)

# Download it مباشرة
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Real clean IDs generation

In [ ]:
import os
import cv2
import json
import pandas as pd

ASSET_DIR = "/content/drive/MyDrive/aadhar_project/assets"
OUTPUT_DIR = "/content/drive/MyDrive/aadhar_project/real_clean"

os.makedirs(OUTPUT_DIR, exist_ok=True)
JSON_DIR = os.path.join(OUTPUT_DIR, "json")
os.makedirs(JSON_DIR, exist_ok=True)

GENDERS = ["male", "female"]

template_img = cv2.imread(f"{ASSET_DIR}/aadhaar card template.jpg")

metadata = []
counter = 1

for gender_folder in GENDERS:

    face_dir = os.path.join(ASSET_DIR, "real_faces", gender_folder)

    face_files = sorted([
        f for f in os.listdir(face_dir)
        if f.lower().endswith(".jpg")
    ])

    gender_hin = "पुरुष" if gender_folder == "male" else "महिला"

    for face_name in face_files:

        face_path = os.path.join(face_dir, face_name)

        # -------- ONE PERSON PER FACE --------
        person = get_matching_record(gender_hin)
        record = {**TEMPLATE_BASE, **person}

        # -------- RENDER CLEAN ONLY --------
        img = template_img.copy()
        render_fields(img, record, face_path)

        # -------- SAVE IMAGE --------
        filename = f"{gender_folder}_{face_name[:-4]}_clean.jpg"
        filepath = os.path.join(OUTPUT_DIR, filename)

        cv2.imwrite(filepath, img)

        # -------- SAVE JSON --------
        json_data = {
            "file": filename,
            "dataset": "real_faces",
            "face": face_name,
            "gender": gender_folder,
            "manipulation": "clean",
            "data": record
        }

        json_path = os.path.join(
            JSON_DIR,
            f"{gender_folder}_{face_name[:-4]}_clean.json"
        )

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=4)

        metadata.append({
            "file": filename,
            "dataset": "real_faces",
            "gender": gender_folder,
            "manipulation": "clean"
        })

        counter += 1

# -------- SAVE CSV --------
labels_df = pd.DataFrame(metadata)
labels_df.to_csv(os.path.join(OUTPUT_DIR, "labels.csv"), index=False)

print("✅ Clean real dataset generated!")
print(f"Total samples: {len(metadata)}")

In [ ]:
import os
import cv2
import json

# ========= PATHS =========
ASSET_DIR = "/content/drive/MyDrive/aadhar_project/assets"
OUTPUT_DIR = "/content/drive/MyDrive/aadhar_project/real_test"

os.makedirs(OUTPUT_DIR, exist_ok=True)
JSON_DIR = os.path.join(OUTPUT_DIR, "json")
os.makedirs(JSON_DIR, exist_ok=True)

# ========= LOAD TEMPLATE =========
template_img = cv2.imread(f"{ASSET_DIR}/aadhaar card template.jpg")

metadata = []
counter = 1

# ========= TARGET =========
GENDERS = ["male", "female"]

for gender_folder in GENDERS:

    face_dir = os.path.join(ASSET_DIR, "real_faces", gender_folder)

    # pick ONLY one image
    face_files = sorted([
        f for f in os.listdir(face_dir)
        if f.lower().endswith(".jpg")
    ])

    if len(face_files) == 0:
        continue

    face_name = face_files[0]   # 👈 only first image
    face_path = os.path.join(face_dir, face_name)

    # map gender
    gender_hin = "पुरुष" if gender_folder == "male" else "महिला"

    # generate ONE identity
    person = get_matching_record(gender_hin)
    record_base = {**TEMPLATE_BASE, **person}

    # apply ALL manipulations
    for manipulation in MANIPULATIONS:

        record = record_base.copy()
        record = manipulation["apply"](record)

        img = template_img.copy()

        render_fields(img, record, face_path)

        # save image
        filename = f"{gender_folder}_{manipulation['label']}.jpg"
        filepath = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(filepath, img)

        # save JSON
        json_data = {
            "file": filename,
            "dataset": "real_faces",
            "gender": gender_folder,
            "manipulation": manipulation["label"],
            "data": record
        }

        json_path = os.path.join(
            JSON_DIR,
            f"{gender_folder}_{manipulation['label']}.json"
        )

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=4)

        metadata.append({
            "file": filename,
            "dataset": "real_faces",
            "gender": gender_folder,
            "manipulation": manipulation["label"]
        })

        counter += 1

